# Chapter 32: Gaussian Mixture Model and EM Algorithm

## Learning Objectives
- Implement E-step and M-step in EM
- Track log-likelihood convergence
- Produce soft cluster assignments
- Analyze initialization sensitivity

## Prerequisites
- Strong understanding of chapters 0-29
- Python and basic linear algebra
- Numpy for probabilistic/deep chapters


# Chapter 32: Gaussian Mixture Model and EM Algorithm

## Why this chapter matters
GMM + EM is a core unsupervised probabilistic algorithm for soft clustering, density modeling, and anomaly detection pipelines.

## Learning goals
1. Derive EM updates for mixture models.
2. Implement E-step responsibilities and M-step parameter updates.
3. Track log-likelihood and convergence.
4. Use GMM for soft assignments and uncertainty-aware clustering.

## Model definition
\[
p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)
\]
Where:
- \(\pi_k\): mixture weights
- \(\mu_k\): component means
- \(\Sigma_k\): component covariances

## EM algorithm
### E-step
Compute responsibilities:
\[
\gamma_{ik} = \frac{\pi_k\mathcal{N}(x_i\mid\mu_k,\Sigma_k)}{\sum_j \pi_j\mathcal{N}(x_i\mid\mu_j,\Sigma_j)}
\]

### M-step
\[
N_k = \sum_i \gamma_{ik}, \quad
\pi_k = \frac{N_k}{N}, \quad
\mu_k = \frac{1}{N_k}\sum_i\gamma_{ik}x_i
\]
\[
\Sigma_k = \frac{1}{N_k}\sum_i \gamma_{ik}(x_i-\mu_k)(x_i-\mu_k)^T
\]

## Practical details
- Add epsilon to covariance diagonals to avoid singularity.
- Initialize with KMeans-style seeds or random samples.
- Use multiple random restarts.

## Industry use cases
- customer segmentation with uncertainty
- probabilistic anomaly scores
- initialization for latent variable models

## Assignment
1. Compare hard KMeans labels vs soft GMM assignments.
2. Evaluate sensitivity to initialization.
3. Add AIC/BIC model selection for component count.


In [ ]:
from __future__ import annotations

import math
from typing import Tuple

import numpy as np


class GaussianMixtureEM:
    def __init__(self, n_components: int = 2, max_iter: int = 100, tol: float = 1e-4, seed: int = 42):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.seed = seed

        self.pi = None
        self.mu = None
        self.var = None  # diagonal covariance (variance vector)

    def _init_params(self, X: np.ndarray) -> None:
        n, d = X.shape
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(n, size=self.n_components, replace=False)

        self.pi = np.full(self.n_components, 1.0 / self.n_components)
        self.mu = X[idx].copy()
        self.var = np.tile(np.var(X, axis=0, keepdims=True), (self.n_components, 1)) + 1e-6

    @staticmethod
    def _gaussian_diag(X: np.ndarray, mu: np.ndarray, var: np.ndarray) -> np.ndarray:
        d = X.shape[1]
        det = np.prod(var)
        norm = 1.0 / math.sqrt((2 * math.pi) ** d * det)
        diff = X - mu
        expo = -0.5 * np.sum((diff ** 2) / var, axis=1)
        return norm * np.exp(expo)

    def _e_step(self, X: np.ndarray) -> np.ndarray:
        n = len(X)
        gamma = np.zeros((n, self.n_components))

        for k in range(self.n_components):
            gamma[:, k] = self.pi[k] * self._gaussian_diag(X, self.mu[k], self.var[k])

        gamma_sum = np.sum(gamma, axis=1, keepdims=True) + 1e-12
        gamma /= gamma_sum
        return gamma

    def _m_step(self, X: np.ndarray, gamma: np.ndarray) -> None:
        n, d = X.shape
        Nk = np.sum(gamma, axis=0) + 1e-12

        self.pi = Nk / n
        self.mu = (gamma.T @ X) / Nk[:, None]

        var = np.zeros((self.n_components, d))
        for k in range(self.n_components):
            diff = X - self.mu[k]
            var[k] = np.sum(gamma[:, [k]] * (diff ** 2), axis=0) / Nk[k]
        self.var = var + 1e-6

    def _log_likelihood(self, X: np.ndarray) -> float:
        probs = np.zeros((len(X), self.n_components))
        for k in range(self.n_components):
            probs[:, k] = self.pi[k] * self._gaussian_diag(X, self.mu[k], self.var[k])
        return float(np.sum(np.log(np.sum(probs, axis=1) + 1e-12)))

    def fit(self, X: np.ndarray) -> "GaussianMixtureEM":
        self._init_params(X)
        prev_ll = -float("inf")

        for _ in range(self.max_iter):
            gamma = self._e_step(X)
            self._m_step(X, gamma)
            ll = self._log_likelihood(X)
            if abs(ll - prev_ll) < self.tol:
                break
            prev_ll = ll

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._e_step(X)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.argmax(self.predict_proba(X), axis=1)


def make_data(seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    a = rng.normal(loc=[0.0, 0.0], scale=[0.5, 0.7], size=(120, 2))
    b = rng.normal(loc=[3.5, 3.0], scale=[0.6, 0.6], size=(120, 2))
    return np.vstack([a, b])


if __name__ == "__main__":
    X = make_data()
    gmm = GaussianMixtureEM(n_components=2, max_iter=200, tol=1e-5)
    gmm.fit(X)
    labels = gmm.predict(X)

    print("Mixture weights:", np.round(gmm.pi, 4))
    print("Means:\n", np.round(gmm.mu, 4))
    print("Cluster counts:", np.bincount(labels))


## Checkpoint

1. You can explain responsibilities in E-step.
2. You can update pi/mu/var in M-step.
3. You can diagnose convergence using likelihood.

## Guided Exercise
Run multiple seeds and compare final means to evaluate local optimum sensitivity.

In [ ]:
# TODO (Student Exercise)
X = make_data()
for seed in [0, 1, 2, 3, 4]:
    gmm = GaussianMixtureEM(n_components=2, max_iter=150, seed=seed)
    gmm.fit(X)
    print('seed', seed, 'means', np.round(gmm.mu, 2))

## Quick Quiz

**Q1.** Why use soft assignments in GMM?

**Answer:** Points can belong to components with uncertainty, not hard labels only.

**Q2.** Why add epsilon to variance?

**Answer:** To avoid singular covariance and numerical instability.

**Q3.** What does EM optimize each iteration?

**Answer:** It monotonically improves (or keeps) data log-likelihood.


## Homework
Add BIC-based component selection and choose the best `K` on synthetic data.